In [ ]:
import os
from google.colab import drive

# Check if /content/drive exists and is not empty, then clear it.
if os.path.exists('/content/drive') and os.listdir('/content/drive'):
    print("Clearing existing files in /content/drive...")
    for item in os.listdir('/content/drive'):
        item_path = os.path.join('/content/drive', item)
        if os.path.isdir(item_path):
            os.system(f'rm -rf {item_path}') # Use shell command for force removal
        else:
            os.remove(item_path)

drive.mount('/content/drive', force_remount=True)

Clearing existing files in /content/drive...
Mounted at /content/drive


In [ ]:
import os
import shutil
import random

In [ ]:
import cv2
import numpy as np
import os
import glob
import random

# ==========================================
# 1. CLASSE PREPROCESSING
# ==========================================
class Preprocessing:
    @staticmethod
    def normalize_stain_macenko(img, Io=240, alpha=1, beta=0.15):
        img_original = img.copy()
        HERef = np.array([[0.5626, 0.2159], [0.7201, 0.8012], [0.4062, 0.5581]])
        maxCRef = np.array([1.9705, 1.0308])

        h, w, c = img.shape
        img_flat = img.reshape((-1, 3))
        OD = -np.log((img_flat.astype(float) + 1) / Io)
        ODhat = OD[~np.any(OD < beta, axis=1)]

        if len(ODhat) < 10: return img_original

        eigvals, eigvecs = np.linalg.eigh(np.cov(ODhat.T))
        Vec = eigvecs[:, 1:3]
        That = np.dot(ODhat, Vec)
        phi = np.arctan2(That[:, 1], That[:, 0])
        minPhi = np.percentile(phi, alpha)
        maxPhi = np.percentile(phi, 100 - alpha)
        vMin = np.dot(Vec, [np.cos(minPhi), np.sin(minPhi)])
        vMax = np.dot(Vec, [np.cos(maxPhi), np.sin(maxPhi)])

        if vMin[0] > vMax[0]: HE = np.array([vMin, vMax])
        else: HE = np.array([vMax, vMin])
        HE = HE.T
        Y = OD.T
        C = np.linalg.lstsq(HE, Y, rcond=None)[0]
        maxC = np.percentile(C, 99, axis=1)
        scale = maxC / maxCRef
        scale[scale == 0] = 1e-5
        C2 = C / scale[:, None]
        Inorm = Io * np.exp(-np.dot(HERef, C2))
        Inorm[Inorm > 255] = 255
        Inorm = Inorm.T.reshape((h, w, 3)).astype(np.uint8)
        return Inorm

    @staticmethod
    def data_augmentation(img, method):
        if method is None:
            method = random.choice(['horizontal', 'vertical', 'rotate90', 'none'])
        if method == 'horizontal': return cv2.flip(img, 1)
        elif method == 'vertical': return cv2.flip(img, 0)
        elif method == 'rotate90': return cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
        else: return img

    # FUNZIONE per augmentare anche MASCHERA e MANUAL_PY
    @staticmethod
    def augment_masks(mask, mask_py, method):
        if method == 'horizontal':
            return cv2.flip(mask, 1), cv2.flip(mask_py, 1)
        elif method == 'vertical':
            return cv2.flip(mask, 0), cv2.flip(mask_py, 0)
        elif method == 'rotate90':
            return (cv2.rotate(mask, cv2.ROTATE_90_CLOCKWISE),
                    cv2.rotate(mask_py, cv2.ROTATE_90_CLOCKWISE))
        return mask, mask_py




# ==========================================
# 2. GESTIONE DATASET 
# ==========================================

def process_and_save_dataset(base_input_path, base_output_path):

    subsets = ['TRS', 'VS', 'TS']
    augmentation_methods = ['horizontal', 'vertical', 'rotate90']

    for subset in subsets:
        print(f"\n--- Elaborazione {subset} ---")

        # cartelle input
        img_folder = os.path.join(base_input_path, subset, "image")
        man_folder = os.path.join(base_input_path, subset, "manual")
        manpy_folder = os.path.join(base_input_path, subset, "manual_py")

        # cartelle output (3 invece di 1)
        out_img  = os.path.join(base_output_path, subset, "image")
        out_man  = os.path.join(base_output_path, subset, "manual")
        out_manpy = os.path.join(base_output_path, subset, "manual_py")

        os.makedirs(out_img, exist_ok=True)
        os.makedirs(out_man, exist_ok=True)
        os.makedirs(out_manpy, exist_ok=True)

        # lista immagini
        image_files = glob.glob(os.path.join(img_folder, "*.png"))

        do_augment = (subset == "TRS")

        for img_path in image_files:
            filename = os.path.basename(img_path)
            base, ext = os.path.splitext(filename)

            # carica immagine
            img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

            # carica maschere
            mask = cv2.imread(os.path.join(man_folder, filename), cv2.IMREAD_GRAYSCALE)
            mask_py = cv2.imread(os.path.join(manpy_folder, filename), cv2.IMREAD_GRAYSCALE)

            # 1️ ORIGINAL PREPROCESSATO — MACENKO ONLY
            preproc_img = Preprocessing.normalize_stain_macenko(img)

            cv2.imwrite(os.path.join(out_img,  f"{base}_preproc{ext}"), preproc_img)
            cv2.imwrite(os.path.join(out_man,  f"{base}_preproc{ext}"), mask)
            cv2.imwrite(os.path.join(out_manpy,f"{base}_preproc{ext}"), mask_py)

            # 2️ AUGMENTATION (solo TRS)
            if do_augment:
                for method in augmentation_methods:

                    # IMMAGINE
                    aug_img = Preprocessing.data_augmentation(preproc_img, method)

                    # MASCHERE
                    aug_mask, aug_mask_py = Preprocessing.augment_masks(mask, mask_py, method)

                    # salva tutto
                    cv2.imwrite(os.path.join(out_img,  f"{base}_aug_{method}{ext}"), aug_img)
                    cv2.imwrite(os.path.join(out_man,  f"{base}_aug_{method}{ext}"), aug_mask)
                    cv2.imwrite(os.path.join(out_manpy,f"{base}_aug_{method}{ext}"), aug_mask_py)

    print("\n FINE! Dataset processato + aumentato con 3 cartelle sincronizzate.")


In [ ]:
INPUT_PATH = '/content/drive/MyDrive/sblao/new_dataset'
OUTPUT_PATH = '/content/drive/MyDrive/sblao/dataset_diviso_PROCESSED'

process_and_save_dataset(INPUT_PATH, OUTPUT_PATH)



--- Elaborazione TRS ---


KeyboardInterrupt: 

In [ ]:
import os
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# ================================================================
# PATH CORRETTI
# ================================================================
base_path = '/content/drive/MyDrive/sblao/dataset_diviso_PROCESSED'

TRS = os.path.join(base_path, "TRS")
TS  = os.path.join(base_path, "TS")
VS  = os.path.join(base_path, "VS")

train_img  = os.path.join(TRS, "image")
train_mask = os.path.join(TRS, "manual")

val_img    = os.path.join(VS, "image")
val_mask   = os.path.join(VS, "manual")

# ================================================================
# CHECKPOINTS
# ================================================================
ckpt_dir = '/content/drive/MyDrive/sblao/checkpoints'
os.makedirs(ckpt_dir, exist_ok=True)
print("Cartella checkpoints:", ckpt_dir)


# ================================================================
# DATASET
# ================================================================
class SteatosisDataset(Dataset):
    def __init__(self, img_dir, mask_dir):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.files = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(".png")])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]

        img = np.array(Image.open(os.path.join(self.img_dir, fname))).astype(np.float32)

        if img.ndim == 2:
            img = np.expand_dims(img, axis=-1)

        img = img / 255.0
        img = torch.from_numpy(img).permute(2,0,1).float()

        mask = np.array(Image.open(os.path.join(self.mask_dir, fname))).astype(np.float32)
        mask = (mask > 0).astype(np.float32)
        mask = torch.from_numpy(mask).unsqueeze(0).float()

        return img, mask


# ================================================================
# SE BLOCK
# ================================================================
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y


# ================================================================
# DOUBLE CONV
# ================================================================
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


# ================================================================
# U-NET 7A (ConvTranspose + SE-block + init_filters=64)
# ================================================================
class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, init_filters=64, depth=3, bilinear=False):
        super().__init__()

        self.depth = depth
        self.pool = nn.MaxPool2d(2)

        self.down_layers = nn.ModuleList()
        self.up_layers = nn.ModuleList()
        self.se_blocks = nn.ModuleList()

        filters = init_filters

        # ENCODER
        for d in range(depth):
            conv = DoubleConv(in_channels, filters)
            self.down_layers.append(conv)
            self.se_blocks.append(SEBlock(filters))
            in_channels = filters
            filters *= 2

        # BOTTLE NECK
        self.bottleneck = DoubleConv(in_channels, filters)

        # DECODER (CONVTRANSPOSE)
        for d in range(depth):
            filters //= 2
            up = nn.ConvTranspose2d(filters * 2, filters, kernel_size=2, stride=2)

            self.up_layers.append(nn.ModuleDict({
                "up": up,
                "conv": DoubleConv(filters * 2, filters)
            }))

        self.out_conv = nn.Conv2d(init_filters, out_channels, 1)

    def forward(self, x):
        skip = []

        # ENCODER
        for down in self.down_layers:
            x = down(x)
            skip.append(x)
            x = self.pool(x)

        # BOTTLE
        x = self.bottleneck(x)

        # DECODER con SE-BLOCK applicato nell'ordine corretto
        for i in range(self.depth):
            s = skip[-(i+1)]                # skip decrescente
            s = self.se_blocks[-(i+1)](s)   # SE-block corrispondente

            x = self.up_layers[i]["up"](x)

            if x.size() != s.size():
                x = F.interpolate(x, size=s.shape[2:])

            x = torch.cat([s, x], dim=1)
            x = self.up_layers[i]["conv"](x)

        return self.out_conv(x)


# ================================================================
# LOSS (BCE + DICE)
# ================================================================
def dice_loss(pred, target, eps=1e-6):
    pred = torch.sigmoid(pred)
    inter = (pred * target).sum()
    den = pred.sum() + target.sum()
    return 1 - (2*inter + eps) / (den + eps)

bce = nn.BCEWithLogitsLoss()

def loss_fn(pred, target):
    return bce(pred, target) + dice_loss(pred, target)


# ================================================================
# DATASET + DATALOADER
# ================================================================
train_ds = SteatosisDataset(train_img, train_mask)
val_ds   = SteatosisDataset(val_img, val_mask)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=4, shuffle=False)

print("Train:", len(train_ds), "immagini")
print("Val:  ", len(val_ds), "immagini")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


# ================================================================
# MODELLO 7A + ADAMW
# ================================================================
model = UNet(in_channels=3, init_filters=64, bilinear=False).to(device)

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-2
)

EPOCHS = 20
best_val_loss = float("inf")
best_epoch = -1


# ================================================================
# TRAINING LOOP
# ================================================================
for epoch in range(1, EPOCHS+1):
    model.train()
    train_loss = 0

    for imgs, masks in tqdm(train_loader, desc=f"Training {epoch}/{EPOCHS}"):
        imgs, masks = imgs.to(device), masks.to(device)

        optimizer.zero_grad()
        out = model(imgs)
        loss = loss_fn(out, masks)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * imgs.size(0)

    train_loss /= len(train_ds)

    # VALIDATION
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            out = model(imgs)
            loss = loss_fn(out, masks)
            val_loss += loss.item() * imgs.size(0)

    val_loss /= len(val_ds)

    print(f"Epoch {epoch}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    # SALVATAGGIO CHECKPOINT
    ckpt_path = os.path.join(ckpt_dir, f"model_epoch_{epoch}.pth")
    torch.save(model.state_dict(), ckpt_path)
    print("Checkpoint salvato:", ckpt_path)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch

print("\n Training COMPLETATO!")
print(f" Epoca migliore: {best_epoch} | Val Loss = {best_val_loss:.6f}")

Cartella checkpoints: /content/drive/MyDrive/sblao/checkpoints
Train: 1592 immagini
Val:   56 immagini
Device: cuda


Training 1/20: 100%|██████████| 398/398 [25:30<00:00,  3.85s/it]


Epoch 1/20 | Train Loss: 1.1658 | Val Loss: 1.3229
Checkpoint salvato: /content/drive/MyDrive/sblao/checkpoints/model_epoch_1.pth


Training 2/20: 100%|██████████| 398/398 [03:46<00:00,  1.76it/s]


Epoch 2/20 | Train Loss: 0.9082 | Val Loss: 0.8784
Checkpoint salvato: /content/drive/MyDrive/sblao/checkpoints/model_epoch_2.pth


...

Epoch 18/20 | Train Loss: 0.3410 | Val Loss: 0.6051
Checkpoint salvato: /content/drive/MyDrive/sblao/checkpoints/model_epoch_18.pth


Training 19/20: 100%|██████████| 398/398 [03:51<00:00,  1.72it/s]


Epoch 19/20 | Train Loss: 0.3396 | Val Loss: 0.6389
Checkpoint salvato: /content/drive/MyDrive/sblao/checkpoints/model_epoch_19.pth


Training 20/20: 100%|██████████| 398/398 [03:53<00:00,  1.71it/s]


Epoch 20/20 | Train Loss: 0.3340 | Val Loss: 0.6082
Checkpoint salvato: /content/drive/MyDrive/sblao/checkpoints/model_epoch_20.pth

🎉 Training COMPLETATO!
🏆 Epoca migliore: 16 | Val Loss = 0.597633


In [ ]:
# ================================================================
# CELLA UNICA: LOAD MODEL + VALIDATION + SWEEP SOGLIA
# ================================================================

# ----------------
# IMPORT
# ----------------
import os
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ----------------
# PATH
# ----------------
base_path = '/content/drive/MyDrive/sblao/dataset_diviso_PROCESSED'
ckpt_dir  = '/content/drive/MyDrive/sblao/checkpoints'

VS = os.path.join(base_path, "VS")
val_img  = os.path.join(VS, "image")
val_mask = os.path.join(VS, "manual")

# ----------------
# DATASET (IDENTICO AL TRAINING)
# ----------------
class SteatosisDataset(Dataset):
    def __init__(self, img_dir, mask_dir):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.files = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(".png")])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]

        img = np.array(Image.open(os.path.join(self.img_dir, fname))).astype(np.float32)
        if img.ndim == 2:
            img = np.expand_dims(img, axis=-1)

        img = img / 255.0
        img = torch.from_numpy(img).permute(2,0,1).float()

        mask = np.array(Image.open(os.path.join(self.mask_dir, fname))).astype(np.float32)
        mask = (mask > 0).astype(np.float32)
        mask = torch.from_numpy(mask).unsqueeze(0).float()

        return img, mask

# ----------------
# SE BLOCK
# ----------------
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y

# ----------------
# DOUBLE CONV
# ----------------
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

# ----------------
# U-NET 7A
# ----------------
class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, init_filters=64, depth=3):
        super().__init__()

        self.depth = depth
        self.pool = nn.MaxPool2d(2)

        self.down_layers = nn.ModuleList()
        self.up_layers = nn.ModuleList()
        self.se_blocks = nn.ModuleList()

        filters = init_filters

        for _ in range(depth):
            self.down_layers.append(DoubleConv(in_channels, filters))
            self.se_blocks.append(SEBlock(filters))
            in_channels = filters
            filters *= 2

        self.bottleneck = DoubleConv(in_channels, filters)

        for _ in range(depth):
            filters //= 2
            self.up_layers.append(nn.ModuleDict({
                "up": nn.ConvTranspose2d(filters * 2, filters, kernel_size=2, stride=2),
                "conv": DoubleConv(filters * 2, filters)
            }))

        self.out_conv = nn.Conv2d(init_filters, out_channels, 1)

    def forward(self, x):
        skip = []

        for down in self.down_layers:
            x = down(x)
            skip.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)

        for i in range(self.depth):
            s = skip[-(i+1)]
            s = self.se_blocks[-(i+1)](s)

            x = self.up_layers[i]["up"](x)
            if x.size() != s.size():
                x = F.interpolate(x, size=s.shape[2:])
            x = torch.cat([s, x], dim=1)
            x = self.up_layers[i]["conv"](x)

        return self.out_conv(x)

# ----------------
# DEVICE + MODEL
# ----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNet(in_channels=3).to(device)

# ----------------
# VALIDATION LOADER
# ----------------
val_ds = SteatosisDataset(val_img, val_mask)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False)

print("Validation images:", len(val_ds))
print("Device:", device)

# ----------------
# CARICAMENTO CHECKPOINT MIGLIORE
# ----------------
best_epoch = 16
ckpt_path = os.path.join(ckpt_dir, f"model_epoch_{best_epoch}.pth")

model.load_state_dict(torch.load(ckpt_path, map_location=device))
model.eval()

print(f" Modello caricato: epoca {best_epoch}")

# ----------------
# DICE PER IMMAGINE
# ----------------
def dice_single(pred, target, eps=1e-6):
    inter = (pred * target).sum()
    den = pred.sum() + target.sum()
    return (2 * inter + eps) / (den + eps)

# ----------------
# SWEEP SOGLIA
# ----------------
thresholds = np.arange(0.05, 0.95, 0.05)
dice_per_threshold = {t: [] for t in thresholds}
excluded_empty = 0

with torch.no_grad():
    for imgs, masks in tqdm(val_loader, desc="Validation sweep soglia"):
        imgs = imgs.to(device)
        masks = masks.to(device)

        probs = torch.sigmoid(model(imgs))

        for i in range(probs.shape[0]):
            gt = masks[i]
            if gt.sum() == 0:
                excluded_empty += 1
                continue

            for t in thresholds:
                bin_pred = (probs[i] > t).float()
                d = dice_single(bin_pred, gt)
                dice_per_threshold[t].append(d.item())

print(f" Immagini con GT vuota escluse: {excluded_empty}")

# ----------------
# RISULTATI FINALI
# ----------------
mean_dice = {t: np.mean(dice_per_threshold[t]) for t in thresholds}
best_threshold = max(mean_dice, key=mean_dice.get)

print("\n Dice medio per soglia (VS, GT non vuote):")
for t in thresholds:
    print(f"  soglia {t:.2f} → Dice = {mean_dice[t]:.4f}")

print("\n SOGLIA OTTIMALE")
print(f" best_threshold = {best_threshold:.2f}")
print(f" Dice medio = {mean_dice[best_threshold]:.4f}")


Validation images: 56
Device: cpu
✅ Modello caricato: epoca 16


Validation sweep soglia: 100%|██████████| 14/14 [05:17<00:00, 22.68s/it]

⚠️ Immagini con GT vuota escluse: 28

📊 Dice medio per soglia (VS, GT non vuote):
  soglia 0.05 → Dice = 0.6556
  soglia 0.10 → Dice = 0.6726
  soglia 0.15 → Dice = 0.6796
  soglia 0.20 → Dice = 0.6824
  soglia 0.25 → Dice = 0.6845
  soglia 0.30 → Dice = 0.6856
  soglia 0.35 → Dice = 0.6873
  soglia 0.40 → Dice = 0.6878
  soglia 0.45 → Dice = 0.6885
  soglia 0.50 → Dice = 0.6893
  soglia 0.55 → Dice = 0.6904
  soglia 0.60 → Dice = 0.6909
  soglia 0.65 → Dice = 0.6906
  soglia 0.70 → Dice = 0.6900
  soglia 0.75 → Dice = 0.6885
  soglia 0.80 → Dice = 0.6880
  soglia 0.85 → Dice = 0.6833
  soglia 0.90 → Dice = 0.6745

🏆 SOGLIA OTTIMALE
👉 best_threshold = 0.60
👉 Dice medio = 0.6909


In [ ]:
import os
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

# ================================================================
# PATH CORRETTI (Rete 7A)
# ================================================================
base_path = '/content/drive/MyDrive/sblao/dataset_diviso_PROCESSED'
TS = os.path.join(base_path, "TS")

test_img_dir = os.path.join(TS, "image")

# Cartella predizioni
pred_dir = os.path.join(base_path, "TEST_PRED")
os.makedirs(pred_dir, exist_ok=True)

# ================================================================
# ARCHITETTURA IDENTICA AL TRAINING DELLA 7A
# ================================================================
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.double_conv(x)


class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y


class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, init_filters=64, depth=3, bilinear=False):
        super().__init__()

        self.depth = depth
        self.pool = nn.MaxPool2d(2)

        self.down_layers = nn.ModuleList()
        self.up_layers   = nn.ModuleList()
        self.se_blocks   = nn.ModuleList()

        filters = init_filters

        # ---------- ENCODER ----------
        for d in range(depth):
            conv = DoubleConv(in_channels, filters)
            self.down_layers.append(conv)
            self.se_blocks.append(SEBlock(filters))

            in_channels = filters
            filters *= 2

        # ---------- BOTTLENECK ----------
        self.bottleneck = DoubleConv(in_channels, filters)

        # ---------- DECODER (ConvTranspose2D) ----------
        for d in range(depth):
            filters //= 2

            up = nn.ConvTranspose2d(filters * 2, filters, kernel_size=2, stride=2)

            self.up_layers.append(nn.ModuleDict({
                "up": up,
                "conv": DoubleConv(filters * 2, filters)
            }))

        self.out_conv = nn.Conv2d(init_filters, out_channels, 1)


    def forward(self, x):
        skip = []

        # -------- ENCODER --------
        for down in self.down_layers:
            x = down(x)
            skip.append(x)
            x = self.pool(x)

        # -------- BOTTLENECK --------
        x = self.bottleneck(x)

        # -------- DECODER + SE --------
        for i in range(self.depth):
            s = skip[-(i+1)]            # skip appropriato
            s = self.se_blocks[-(i+1)](s)  # SE-block corretto

            x = self.up_layers[i]["up"](x)

            # Se mismatch di dimensioni, correggi
            if x.size() != s.size():
                x = F.interpolate(x, size=s.shape[2:])

            x = torch.cat([s, x], dim=1)
            x = self.up_layers[i]["conv"](x)

        return self.out_conv(x)


# ================================================================
# CARICAMENTO MODELLO MIGLIORE
# ================================================================
best_epoch = 16   # <---- cambia con l'epoca migliore

ckpt_path = f"/content/drive/MyDrive/sblao/checkpoints/model_epoch_{best_epoch}.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet(in_channels=3, init_filters=64, bilinear=False).to(device)
model.load_state_dict(torch.load(ckpt_path, map_location=device))
model.eval()

print(f" Modello 7A caricato (epoca {best_epoch})")


# ================================================================
# INFERENCE
# ================================================================
test_files = sorted([f for f in os.listdir(test_img_dir) if f.endswith(".png")])

for fname in tqdm(test_files, desc="Inference 7A"):
    path = os.path.join(test_img_dir, fname)

    img = np.array(Image.open(path)).astype(np.float32)

    # Se grayscale → aggiungi canale
    if img.ndim == 2:
        img = np.expand_dims(img, axis=-1)

    img = img / 255.0

    img_t = torch.from_numpy(img).permute(2,0,1).unsqueeze(0).float().to(device)

    # Predizione
    with torch.no_grad():
        pred = model(img_t)
        pred = torch.sigmoid(pred)
        pred = (pred > best_threshold).float()

    pred_np = pred.squeeze().cpu().numpy() * 255
    Image.fromarray(pred_np.astype(np.uint8)).save(os.path.join(pred_dir, fname))

print(" Inference 7A completata")
print(" Predizioni salvate in:", pred_dir)

✅ Modello 7A caricato (epoca 16)


Inference 7A: 100%|██████████| 56/56 [05:12<00:00,  5.58s/it]

🎉 Inference 7A completata
📁 Predizioni salvate in: /content/drive/MyDrive/sblao/dataset_diviso_PROCESSED/TEST_PRED
